In [ ]:
# CELL 1: Environment, paths, and reproducibility setup
import json
import pandas as pd
import chromadb
import re
from pathlib import Path
from chromadb.utils import embedding_functions
from llama_cpp import Llama, LlamaGrammar

RESULTS_DIR = Path("../data/results")
ATTCK_DIR = Path("../data/attck")
CHROMA_DIR = Path("../data/chroma")
STIX_FILE = ATTCK_DIR / "enterprise-attack.json"

COMMUNITY_FILE = RESULTS_DIR / "community_assignments.csv"
TRIPLES_FILE = RESULTS_DIR / "community_triples.json"
REPORTS_FILE = RESULTS_DIR / "stage5_rag_reports.csv"
METRICS_FILE = RESULTS_DIR / "stage5_rag_metrics.json"

EMBED_MODEL = "all-MiniLM-L6-v2"
MODEL_PATH = "./qwen2.5-3b-instruct-q4_k_m.gguf"
TOP_K = 3
RANDOM_SEED = 42

In [ ]:
# CELL 2: Load community assignments and semantic triples
community_df = pd.read_csv(COMMUNITY_FILE, low_memory=False)
with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

# Filter out benign traffic for evaluation
eval_df = community_df[community_df["attck_technique_id"] != "BENIGN"].copy()
eval_cids = eval_df["community_id"].unique()[:20]  # Cap at 20 for initial validation
print(f"Loaded {len(eval_df)} rows | Evaluating {len(eval_cids)} communities")

In [ ]:
# CELL 3: Initialize ChromaDB with ATT&CK STIX bundle
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name="attack_techniques", embedding_function=ef)

if collection.count() == 0:
    with open(STIX_FILE, "r", encoding="utf-8") as f:
        bundle = json.load(f)
    docs, ids, metas = [], [], []
    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern" or obj.get("revoked"):
            continue
        tid = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tid = ref.get("external_id")
                break
        if not tid: continue
        tactics = [p["phase_name"].replace("-", " ").title() for p in obj.get("kill_chain_phases", []) if p.get("kill_chain_name")=="mitre-attack"]
        text = f"ID: {tid}\nName: {obj.get('name')}\nTactic: {', '.join(tactics)}\nDescription: {obj.get('description', '')}"
        docs.append(text)
        ids.append(tid)
        metas.append({"technique_id": tid, "name": obj.get("name"), "tactic": ", ".join(tactics)})
    collection.add(ids=ids, documents=docs, metadatas=metas)
    print(f"Inserted {collection.count()} ATT&CK procedures into ChromaDB")
else:
    print(f"ChromaDB ready: {collection.count()} documents")

In [ ]:
# CELL 4: LLM initialization and grammar constraint for reports
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "technique_id": {"type": "string"},
        "summary": {"type": "string"},
        "evidence": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["technique_id", "summary", "evidence", "next_step"]
}
report_grammar = LlamaGrammar.from_json_schema(json.dumps(REPORT_SCHEMA))

llm = Llama(
    model_path=MODEL_PATH, n_ctx=2048, n_gpu_layers=-1, n_threads=8,
    verbose=False, seed=RANDOM_SEED, temperature=0
)
print("LLM & report grammar initialized")

In [ ]:
# CELL 5: Retrieval query builder (NO ground truth leakage) and prompt templates
def build_retrieval_query(cid):
    group = community_df[community_df["community_id"] == cid]
    ports = ", ".join([str(p) for p in group["Destination Port"].value_counts().head(3).index])
    triples = community_triples.get(str(cid), [])
    relations = "; ".join([f"{t['subject']} {t['relation']} {t['target']}" for t in triples[:5]])
    # Professor requirement: retrieval driven ONLY by community summaries/triples
    return f"Ports: {ports}. Relations: {relations}."

def build_prompt(summary, context=None):
    ctx_block = f"\nRETRIEVED ATT&CK CONTEXT:\n{context}" if context else ""
    instruction = "You are a SOC analyst. Write a structured incident report. Use ONLY the provided summary and context. Do NOT invent techniques. Output valid JSON matching the schema."
    return f"[INST] {instruction}\nSUMMARY:\n{summary}{ctx_block}\n[/INST]"

In [ ]:
# CELL 6: Deterministic report generation function
def generate_report(summary, context=None, mode="rag"):
    ctx = context if mode=="rag" else None
    prompt = build_prompt(summary, ctx)
    
    out = llm(
        prompt, max_tokens=300, temperature=0, seed=RANDOM_SEED,
        grammar=report_grammar, stop=["[/INST]"]
    )
    raw = out["choices"][0]["text"].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"technique_id": "Unknown", "summary": summary[:50], "evidence": "none", "next_step": "investigate"}

# LLM warmup to stabilize CUDA context and prevent cold-start truncation
print("Warming up LLM context...")
_ = llm("[INST] Warmup.[/INST]", max_tokens=4, temperature=0, seed=RANDOM_SEED, grammar=report_grammar)
print("Warmup complete.")

In [ ]:
# CELL 7: Evaluation loop (Baseline vs RAG)
results = []

def extract_ids(text):
    return re.findall(r"T\d{4}(?:\.\d{3})?", str(text) if text else "")

for cid in eval_cids:
    group = eval_df[eval_df["community_id"] == cid]
    gt = group["attck_technique_id"].mode().iloc[0]
    summary = build_retrieval_query(cid)
    
    # RAG retrieval
    query_res = collection.query(query_texts=[summary], n_results=TOP_K)
    docs = query_res["documents"][0]
    metas = query_res["metadatas"][0]
    ctx = "\n\n".join(docs)
    
    rag_report = generate_report(summary, ctx, mode="rag")
    base_report = generate_report(summary, mode="baseline")
    
    results.append({
        "community_id": cid,
        "ground_truth": gt,
        "retrieved_ids": [m.get("technique_id") for m in metas],
        "baseline_technique_id": base_report.get("technique_id", ""),
        "rag_technique_id": rag_report.get("technique_id", ""),
        "baseline_match": gt in extract_ids(base_report.get("technique_id") + base_report.get("summary")),
        "rag_match": gt in extract_ids(rag_report.get("technique_id") + rag_report.get("summary"))
    })
print(f"Evaluation complete for {len(results)} communities")